Business Question

Management asks:

Who are our top customers?

for the above question We need:

Revenue per customer
Number of orders
Average order value
Top 20 customers

in these analyatics we will be covering all the above tops which can answer the question 

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    sum,
    countDistinct,
    round,
    desc
)

In [6]:
spark = ( 
    SparkSession.builder
    .appName("Customer_Analytics")
    .master("local[*]")
    .getOrCreate()

)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/12 18:45:18 WARN Utils: Your hostname, MacBook-pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.174 instead (on interface en0)
26/06/12 18:45:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 18:45:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/12 18:45:19 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [8]:
# reading Sales fact table 
sales_df=spark.read.parquet(
    "output/gold/sales_fact"
)
print("Total Sales Records:")
print(sales_df.count())
sales_df.printSchema()

Total Sales Records:
117601
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- purchase_timestamp: timestamp (nullable = true)
 |-- purchase_year: integer (nullable = true)
 |-- purchase_month: integer (nullable = true)
 |-- purchase_day: integer (nullable = true)
 |-- purchase_quarter: integer (nullable = true)
 |-- purchase_weekday: string (nullable = true)



In [15]:
# Revenue by Customer
#Calculates Customer Lifetime Value (CLV) metrics per customer:
# 1. Groups data by customer_id
# 2. Computes total revenue (sum of payments rounded to 2 decimals)
# 3. Counts unique orders placed (ignoring multiple items per order)
customer_revenue_df=(
    sales_df.groupBy("customer_id")
    .agg(
        round(sum("payment_value"),2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders")
    )
)
customer_revenue_df.show(15)
customer_revenue_df.printSchema()

+--------------------+-------------+------------+
|         customer_id|total_revenue|total_orders|
+--------------------+-------------+------------+
|a836f6725983cd799...|       106.84|           1|
|72ecfc69f7d90359d...|       185.35|           1|
|dca00fb1b6171b713...|        69.64|           1|
|6f804e6a8f98ba0d1...|        47.59|           1|
|177fc7ae2e9f2151a...|        73.01|           1|
|57d74bba7d8b5157d...|         26.0|           1|
|60a27a1b80babef80...|       165.92|           1|
|9a5d3906304996948...|       243.56|           1|
|36a1aa63bf2ebcd49...|         98.8|           1|
|1187acd131fe97cfc...|        46.11|           1|
|088935be5a63fd416...|        83.34|           1|
|80447be02d8f3f3f3...|         41.0|           1|
|87bc0b31db47421f9...|      1153.25|           1|
|c920352ccf11e554a...|        75.07|           1|
|a3f3f06060fe4f8e8...|       178.42|           1|
+--------------------+-------------+------------+
only showing top 15 rows
root
 |-- customer_id: st

In [14]:
# Average Order Value

customer_revenue_df=(
    customer_revenue_df.withColumn("avg_order_value",round(customer_revenue_df.total_revenue /
                                                           customer_revenue_df.total_orders, 2))
                                                        
)
customer_revenue_df.show(15)
customer_revenue_df.printSchema()

+--------------------+-------------+------------+---------------+
|         customer_id|total_revenue|total_orders|avg_order_value|
+--------------------+-------------+------------+---------------+
|a836f6725983cd799...|       106.84|           1|         106.84|
|72ecfc69f7d90359d...|       185.35|           1|         185.35|
|dca00fb1b6171b713...|        69.64|           1|          69.64|
|6f804e6a8f98ba0d1...|        47.59|           1|          47.59|
|177fc7ae2e9f2151a...|        73.01|           1|          73.01|
|57d74bba7d8b5157d...|         26.0|           1|           26.0|
|60a27a1b80babef80...|       165.92|           1|         165.92|
|9a5d3906304996948...|       243.56|           1|         243.56|
|36a1aa63bf2ebcd49...|         98.8|           1|           98.8|
|1187acd131fe97cfc...|        46.11|           1|          46.11|
|088935be5a63fd416...|        83.34|           1|          83.34|
|80447be02d8f3f3f3...|         41.0|           1|           41.0|
|87bc0b31d

In [17]:
#top Customers 

top_customers_df =(
    customer_revenue_df.orderBy(desc("total_revenue"))
)
print("\nTop Customers")

top_customers_df.show(
    20,
    truncate=False
)
top_customers_df.printSchema()



Top Customers
+--------------------------------+-------------+------------+
|customer_id                     |total_revenue|total_orders|
+--------------------------------+-------------+------------+
|1617b1357756262bfa56ab541c47bc16|109312.64    |1           |
|bd5d39761aa56689a265d95d8d32b8be|45256.0      |1           |
|be1b70680b9f9694d8c70f41fa3dc92b|44048.0      |1           |
|05455dfa7cd02f13d132aa7a6a9729c6|36489.24     |1           |
|1ff773612ab8934db89fd5afa8afe506|30186.0      |1           |
|ec5b2ba62e574342386871631fafd3fc|29099.52     |1           |
|e7d6802668de6e74d0d6c56565bf2a24|22346.6      |1           |
|8c20d9bfbc96c5d39025d77a3ba83d7f|21874.05     |1           |
|f7622098214b4634b7fe7eee269b5426|19457.04     |1           |
|71901689c5f3e5adc27b1dd16b33f0b8|19174.38     |1           |
|cb87122c4871e202777cf243fbea2d12|18667.0      |1           |
|10de381f8a8d23fff822753305f71cae|18384.75     |1           |
|91f92cfee46b79581b05aa974dd57ce5|17786.88     |1      